# 润色模型对比

用 **dito 的真实润色提示词**（`crates/dito-core/src/prompts.rs`，General 风格）跑 llama.cpp（与 dito 同源运行时，Metal），对比四个 GGUF 候选。

权重在 `models/polish/`（首次需下载，见 README）。

结论速览：
- `qwen3-1.7b`：**不可用**——hybrid thinking 模型，输出全是 `<think>` 推理文本（dito 目录里的这个条目有同样问题）
- `qwen3-4b`（dito 现默认）：稳，0.3–0.5s，但中英混说会把英文词**归化成中文**（quick prototype→快速原型）
- `qwen3.5-2b`：最快 0.25–0.3s，**保留英文术语**（对开发者更友好），英语口语偶尔轻微加戏
- `qwen3.5-4b`：质量最稳 0.5–0.6s，**词典注入完美**——片假名转写按词表全部转回拉丁词，与 ASR 热词形成完整链路

In [1]:
from asr_lab.polish import Polisher

DIRTY = [
    ("ja", "sense空格", "明日の午後3時に 会議室 で 新機能 について話し合います。"),
    ("ja", "口语改口", "えーと、あの、明日の件なんですけど、その、三時じゃなくて三時半で会議室を予約しといてくれますか、お願いします"),
    ("ja", "片假名混说", "このイシュは次のスプリントでフィックスする予定なので、まずクイックなレビューをお願いします。"),
    ("zh", "whisper半角", "今天下午3点,我们在会议室讨论新版本的功能设计,然后把结果同步给测试团队。"),
    ("zh", "口语中英混", "嗯 那个 这个需求的话 就是先做一个 quick prototype 吧 然后我们再 review 一下 哦对了记得同步给测试组"),
    ("en", "英语口语", "um so let's ship this tomorrow morning I think and yeah sync the results with the QA team"),
]

for key in ["qwen3-4b", "qwen3.5-2b", "qwen3.5-4b"]:
    p = Polisher(key)
    p.polish("テストです", "ja")  # warmup
    print(f"\n===== {key} =====")
    for lang, name, text in DIRTY:
        r = p.polish(text, lang)
        print(f"[{name}] {r['elapsed_s']:.2f}s {r['tok_s']:.0f}t/s\n  out: {r['text']}")
    del p


===== qwen3-4b =====


[sense空格] 0.34s 56t/s
  out: 明日の午後3時、会議室で新機能について話し合います。


[口语改口] 0.36s 49t/s
  out: 明日の会議室の予約を、3時半にしてくれますか。


[片假名混说] 0.47s 60t/s
  out: このイシューは次のスプリントでフィックスする予定なので、まずはクイックなレビューをお願いします。


[whisper半角] 0.48s 40t/s
  out: 今天下午3点在会议室讨论新版本的功能设计，并将结果同步给测试团队。


[口语中英混] 0.32s 46t/s
  out: 先做一个快速原型，然后进行评审，记得同步给测试组。


... (11 outputs left out)

[片假名混说] 0.62s 32t/s
  out: このイッシュは次のスプリントで修正する予定なので、まずは簡潔なレビューをお願いします。


[whisper半角] 0.52s 33t/s
  out: 今天下午三点，我们在会议室讨论新版本的功能设计，并将结果同步给测试团队。


[口语中英混] 0.62s 32t/s
  out: 这个需求先做一个快速原型，然后我们再评审一下。对了，记得同步给测试组。


[英语口语] 0.48s 31t/s
  out: Let's ship this tomorrow morning and sync the results with the QA team.


## 词典注入：片假名 → 拉丁词的收尾链路

ASR 层（Qwen3-ASR）对日英混说倾向输出片假名；把用户的英文词表作为 dictionary 传给润色层，`qwen3.5-4b` 能全部转回——这是 ASR 热词 → 润色词典的完整闭环。

In [2]:
p = Polisher("qwen3.5-4b")
text = "このイシュは次のスプリントでフィックスする予定なので、まずクイックなレビューをお願いします。"
print("无词典:", p.polish(text, "ja")["text"])
print("有词典:", p.polish(text, "ja", dictionary=["issue", "sprint", "fix", "quick", "review"])["text"])

无词典: このイッシュは次のスプリントで修正する予定なので、まずは簡潔なレビューをお願いします。


有词典: この issue は次の sprint で fix する予定なので、まず quick な review をお願いします。


## MLX 运行时与更大型号（第二轮）

`MlxPolisher` 走 mlx-lm（Apple 原生）。结论：同权重 MLX 比 llama.cpp 快（9B 快 ~15%）；**9B@4bit 是甜点**（词典注入生效、ja 敬体最佳，MLX 0.8s/条）；9B Q3 不省时间；Qwen3.8-27B（2.7s/条）无质量优势。注意 Qwen3.5 的 MLX 模板默认开 thinking，`MlxPolisher` 已显式关闭。

In [3]:
from asr_lab.polish import MlxPolisher

p = MlxPolisher("qwen3.5-9b-4bit")
text = "このイシュは次のスプリントでフィックスする予定なので、まずクイックなレビューをお願いします。"
print("无词典:", p.polish(text, "ja")["text"])
print("有词典:", p.polish(text, "ja", dictionary=["issue", "sprint", "fix", "quick", "review"])["text"])

/Users/fang2hou/Developer/projects/dito/asr-lab/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


无词典: このイシューは次のスプリントで修正する予定なので、まずはクイックなレビューをお願いします。


有词典: この issue は次の sprint で fix する予定なので、まず quick な review をお願いします。
